# Capítulo 3 · Actividad: análisis de datos con NumPy y pandas

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/riosinda/ml-course/blob/main/notebooks/exercises/chapter_03/01_actividad_numpy_pandas.ipynb)

En esta actividad haces de **analista de datos**: vas a responder preguntas sobre datos reales usando solo **NumPy** y **pandas**, sin gráficas. Lo importante no es solo que el código funcione, sino que **sepas interpretar los resultados**.

### Antes de empezar

1. **Copia esta notebook a `workspace/`** y trabaja sobre la copia. Si editas el original tendrás conflictos al sincronizar tu fork.
2. Si todavía no tienes los datos, descárgalos desde la raíz del repo:
   ```bash
   uv run python scripts/download_data.py
   ```
3. Ten a mano las notebooks de clase: `notebooks/lectures/chapter_03/01_numpy.ipynb` y `02_pandas.ipynb`.

### Cómo trabajar

- Cada ejercicio tiene una celda `# Tu código aquí`. Puedes añadir todas las celdas que necesites.
- En las celdas ✍️ responde con tus palabras y **apoya tus respuestas con números** de tus resultados.
- Las 💡 **pistas** te recuerdan funciones. Si una función no salió en clase, la pista explica cómo usarla.
- Al terminar, ejecuta **Kernel → Restart & Run All** y comprueba que todo funciona de arriba abajo.

| Parte | Datos | Herramienta |
|---|---|---|
| 1. La cafetería | ventas de un mes (generadas en la notebook) | NumPy |
| 2. Las propinas | `data/tips.csv` | pandas (+ un poco de NumPy) |
| Reto opcional | `data/titanic.csv` | pandas |

In [7]:
from pathlib import Path

import numpy as np
import pandas as pd

# Carpeta data/ en la raíz del repo: subimos de carpeta en carpeta hasta dar con pyproject.toml
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
print("Carpeta de datos:", DATA_DIR)

Carpeta de datos: /home/nicebunny/ml-course/data


---
## Parte 1 · La cafetería ☕ (NumPy)

Una cafetería vende 5 productos y ha registrado cuántas unidades vendió de cada uno durante **4 semanas** (28 días, de lunes a domingo). La dueña quiere saber cuánto gana, qué días van mejor y qué producto le conviene más.

Ejecuta la siguiente celda para generar los datos. **No hace falta que la modifiques ni que entiendas cada línea.**

In [8]:
# Datos de la cafetería (no hace falta modificar esta celda)
rng = np.random.default_rng(42)

productos = np.array(["café", "latte", "té", "croissant", "sándwich"])
precios = np.array([1.5, 2.8, 1.2, 2.0, 4.5])       # precio de cada producto, en $

# Unidades vendidas cada día: los fines de semana se vende un 50% más
dia_semana = np.tile(np.arange(7), 4)                # 0 = lunes, ..., 6 = domingo
factor = np.where(dia_semana >= 5, 1.5, 1.0)
promedio = np.outer(factor, [30, 22, 12, 18, 9])     # ventas esperadas por día y producto
ventas_planas = rng.poisson(promedio).ravel()         # ¡todo en una sola fila!

print(ventas_planas[:10], "...")
print("Número de valores:", ventas_planas.size)

[35 28  7 16 13 29 16 10 18 11] ...
Número de valores: 140


### Ejercicio 1 · Dar forma a los datos

`ventas_planas` tiene 140 números seguidos: primero las ventas de los 5 productos del día 1, después las del día 2, y así sucesivamente.

a) Crea una matriz `ventas` con forma `(28, 5)`: una **fila por día** y una **columna por producto** (en el mismo orden que `productos`).

b) Imprime su `shape`, `ndim`, `size` y `dtype`.

c) ¿Cuántos cafés se vendieron el primer día? Comprueba que coincide con `ventas_planas[0]`.

> 💡 **Pista:** `reshape`.

In [9]:
ventas = ventas_planas.reshape(28, 5)
print('shape:', ventas.shape, 'ndim:', ventas.ndim, 'size:', ventas.size, 'dtype:', ventas.dtype)
print('Cafés del primer día:', ventas[0, 0])
print('Coincide:', ventas[0, 0] == ventas_planas[0])

shape: (28, 5) ndim: 2 size: 140 dtype: int64
Cafés del primer día: 35
Coincide: True


### Ejercicio 2 · Indexación

Obtén, usando índices y *slicing* sobre `ventas`:

a) Las ventas de todos los productos el **primer día**.

b) Las ventas de **latte** (columna 1) durante todo el mes.

c) Las ventas de la **última semana** (los últimos 7 días). Comprueba que su forma es `(7, 5)`.

d) Las ventas de **croissant y sándwich** (las dos últimas columnas) durante la **primera semana**.

> 💡 **Pista:** `M[filas, columnas]`. Recuerda que los índices negativos cuentan desde el final.

In [10]:
print('a) Primer día:', ventas[0, :])
print('b) Lattes del mes:', ventas[:, 1])
ultima_semana = ventas[-7:, :]
print('c) Última semana:\n', ultima_semana)
print('Forma:', ultima_semana.shape)
print('d) Croissant y sándwich, primera semana:\n', ventas[:7, -2:])

a) Primer día: [35 28  7 16 13]
b) Lattes del mes: [28 16 17 21 17 29 37 16 23 25 22 19 29 39 24 19 18 22 30 44 27 14 31 21
 30 26 32 45]
c) Última semana:
 [[22 14 11 20  9]
 [34 31 12 27  7]
 [26 21 17 23  7]
 [35 30  3  9 10]
 [23 26 16 21  8]
 [38 32 20 21 11]
 [49 45 18 27  9]]
Forma: (7, 5)
d) Croissant y sándwich, primera semana:
 [[16 13]
 [18 11]
 [17 10]
 [15  8]
 [17  8]
 [22 14]
 [29 12]]


### Ejercicio 3 · Ingresos

a) Calcula la matriz `ingresos` (forma `(28, 5)`): lo que se ingresó por cada producto cada día.

b) ¿Cuánto ingresó la cafetería en todo el mes?

c) Crea `ingreso_diario`, un array con el ingreso total de **cada día** (forma `(28,)`).

d) ¿Cuál fue el ingreso diario medio, el máximo y el mínimo?

> 💡 **Pistas:**
> - `ventas * precios` multiplica **cada fila** de `ventas` por el vector `precios`, elemento a elemento: la columna del café por 1.5, la del latte por 2.8… (este mecanismo se llama *broadcasting*).
> - `M.sum()` suma todo. `M.sum(axis=1)` suma **cada fila** y devuelve un resultado por fila; `M.sum(axis=0)` suma **cada columna**.

In [11]:
ingresos = ventas * precios
print('a) Forma:', ingresos.shape)
print('b) Ingreso total:', round(ingresos.sum(), 2))
ingreso_diario = ingresos.sum(axis=1)
print('c) Ingreso diario:', ingreso_diario.round(2), 'Forma:', ingreso_diario.shape)
print('d) Media, máximo y mínimo:', ingreso_diario.mean().round(2), ingreso_diario.max().round(2), ingreso_diario.min().round(2))

a) Forma: (28, 5)
b) Ingreso total: 6342.5
c) Ingreso diario: [229.8 185.8 194.1 179.1 178.2 291.7 319.1 188.9 183.2 198.3 204.6 191.3
 320.7 317.5 181.9 174.1 176.6 196.8 229.  337.3 279.9 165.9 237.7 195.7
 203.1 204.5 262.1 315.6] Forma: (28,)
d) Media, máximo y mínimo: 226.52 337.3 165.9


### Ejercicio 4 · Días buenos y días flojos

Usa este array con el número de cada día (del 1 al 28):
```python
dias = np.arange(1, 29)
```

a) Crea una máscara con los días cuyo ingreso supera el **ingreso diario medio** y muestra **qué días** fueron.

b) ¿**Cuántos** días fueron?

c) Crea un array que diga `"bueno"` o `"flojo"` para cada día, según si supera o no la media.

d) Calcula `dias[mascara] % 7` (el resto de dividir entre 7). El día 1 fue lunes, así que un resto de **6** es sábado y un resto de **0** es domingo. ¿Qué tienen en común los días buenos?

> 💡 **Pistas:** `.sum()` sobre una máscara cuenta los `True`, porque `True` vale 1 y `False` vale 0. Para c), `np.where`.

In [12]:
dias = np.arange(1, 29)
mascara = ingreso_diario > ingreso_diario.mean()
print('a) Días buenos:', dias[mascara])
print('b) Cantidad:', mascara.sum())
etiquetas = np.where(mascara, 'bueno', 'flojo')
print('c) Etiquetas:', etiquetas)
print('d) Restos:', dias[mascara] % 7)

a) Días buenos: [ 1  6  7 13 14 19 20 21 23 27 28]
b) Cantidad: 11
c) Etiquetas: ['bueno' 'flojo' 'flojo' 'flojo' 'flojo' 'bueno' 'bueno' 'flojo' 'flojo'
 'flojo' 'flojo' 'flojo' 'bueno' 'bueno' 'flojo' 'flojo' 'flojo' 'flojo'
 'bueno' 'bueno' 'bueno' 'flojo' 'bueno' 'flojo' 'flojo' 'flojo' 'bueno'
 'bueno']
d) Restos: [1 6 0 6 0 5 6 0 2 6 0]


✍️ **Tu respuesta:** 

Los 11 días por encima de la media son [1, 6, 7, 13, 14, 19, 20, 21, 23, 27, 28]. Los 8 días de fin de semana superan la media, pero también lo hacen 3 días entre semana: el 1 (lunes), el 19 (viernes) y el 23 (martes). Los días buenos se concentran en fin de semana, aunque no son exclusivamente de fin de semana. En la generación de datos se asignó a los fines de semana un 50% más de ventas esperadas; las ventas efectivas varían aleatoriamente.

### Ejercicio 5 · Análisis por semanas

a) Reorganiza `ingreso_diario` en una matriz `por_semana` de forma `(4, 7)`: una fila por semana y una columna por día (lunes a domingo).

b) ¿Cuánto se ingresó **cada semana**?

c) ¿Cuál es el ingreso medio de **cada día de la semana**? ¿Qué día se vende más? Usa:
```python
nombres_dias = np.array(["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"])
```

d) Compara el ingreso medio de un día de **fin de semana** con el de un día **entre semana**. ¿Cuántas veces más se ingresa en fin de semana?

> 💡 **Pistas:** `axis=1` da un resultado por fila (por semana) y `axis=0` uno por columna (por día de la semana). `arr.argmax()` devuelve la **posición** del valor máximo; úsala como índice en `nombres_dias`.

In [13]:
por_semana = ingreso_diario.reshape(4, 7)
print('a) Forma:', por_semana.shape)
print('b) Totales semanales:', por_semana.sum(axis=1).round(2))
nombres_dias = np.array(['lunes', 'martes', 'miércoles', 'jueves', 'viernes', 'sábado', 'domingo'])
media_dias = por_semana.mean(axis=0)
print('c) Medias por día:', list(zip(nombres_dias, media_dias.round(2))))
print('Mayor ingreso medio:', nombres_dias[media_dias.argmax()])
media_finde = por_semana[:, 5:].mean()
media_laborables = por_semana[:, :5].mean()
print('d) Fin de semana:', round(media_finde, 2), 'Entre semana:', round(media_laborables, 2))
print('Razón:', round(media_finde / media_laborables, 4))

a) Forma: (4, 7)
b) Totales semanales: [1577.8 1604.5 1575.6 1584.6]
c) Medias por día: [(np.str_('lunes'), np.float64(191.62)), (np.str_('martes'), np.float64(195.2)), (np.str_('miércoles'), np.float64(191.18)), (np.str_('jueves'), np.float64(195.9)), (np.str_('viernes'), np.float64(200.75)), (np.str_('sábado'), np.float64(302.95)), (np.str_('domingo'), np.float64(308.02))]
Mayor ingreso medio: domingo
d) Fin de semana: 305.49 Entre semana: 194.93
Razón: 1.5672


### Ejercicio 6 · El producto estrella

a) Calcula el ingreso total de **cada producto** en el mes.

b) ¿Qué **porcentaje** del ingreso total aporta cada producto? Redondea a 1 decimal.

c) ¿Qué producto genera **más dinero**? ¿Y cuál vende **más unidades**?

> 💡 **Pista:** `axis=0` y `argmax`, igual que en el ejercicio anterior.

In [14]:
ingreso_producto = ingresos.sum(axis=0)
unidades_producto = ventas.sum(axis=0)
pct_producto = ingreso_producto / ingresos.sum() * 100
for p, u, i, pct in zip(productos, unidades_producto, ingreso_producto, pct_producto):
    print(f'{p}: {u} unidades, ${i:.2f}, {pct:.1f}%')
print('Más ingresos:', productos[ingreso_producto.argmax()])
print('Más unidades:', productos[unidades_producto.argmax()])

café: 989 unidades, $1483.50, 23.4%
latte: 721 unidades, $2018.80, 31.8%
té: 381 unidades, $457.20, 7.2%
croissant: 557 unidades, $1114.00, 17.6%
sándwich: 282 unidades, $1269.00, 20.0%
Más ingresos: latte
Más unidades: café


✍️ **Pregunta:** ¿por qué el producto que más se vende no es el que más dinero deja? Si fueras la dueña, ¿qué harías con esta información?

✍️ **Tu respuesta:** 

El café vende más unidades (989), pero cuesta $1.50. El latte cuesta $2.80 y aporta $2018.80, frente a $1483.50 del café. Mantendría suficiente café para cubrir la demanda y evaluaría promociones del latte. Para decidir qué conviene más necesito conocer los costos: mayor ingreso no significa mayor utilidad.

---
## Parte 2 · Las propinas 💵 (pandas)

Un camarero anotó, durante varias semanas, cada mesa que atendió: cuánto pagaron, cuánta propina dejaron y algunos datos de la mesa. Queremos responder una pregunta: **¿de qué depende la propina?**

El archivo es `data/tips.csv` y tiene estas columnas:

| Columna | Significado |
|---|---|
| `total_bill` | importe de la cuenta, en $ |
| `tip` | propina, en $ |
| `sex` | sexo de quien pagó (`Female` / `Male`) |
| `smoker` | si en la mesa había fumadores (`Yes` / `No`) |
| `day` | día de la semana (`Thur`, `Fri`, `Sat`, `Sun`) |
| `time` | `Lunch` (comida) o `Dinner` (cena) |
| `size` | número de personas en la mesa |

### Ejercicio 7 · Primer vistazo

a) Carga el archivo en un DataFrame llamado `propinas`.

b) Explóralo con `shape`, `head`, `info` y `describe`.

c) ¿Cuántas mesas hay de cada día? ¿Y de cada momento (comida/cena)?

In [15]:
propinas = pd.read_csv(DATA_DIR / 'tips.csv')
print('Forma:', propinas.shape)
print(propinas.head())
propinas.info()
print(propinas.describe())
print('Mesas por día:\n', propinas['day'].value_counts())
print('Mesas por momento:\n', propinas['time'].value_counts())
print('Faltantes:\n', propinas.isna().sum())

FileNotFoundError: [Errno 2] No such file or directory: '/home/nicebunny/ml-course/data/tips.csv'

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿cuántas mesas hay? ¿Falta algún dato? ¿Cuál fue la cuenta más alta y cuál la mesa más grande?

✍️ **Tu respuesta:** 

Hay 244 mesas y 7 columnas. No hay datos faltantes. La cuenta máxima es $50.81 y la mesa más grande tiene 6 personas. Hay 87 mesas de sábado, 76 de domingo, 62 de jueves y 19 de viernes; 176 cenas y 68 comidas.

### Ejercicio 8 · Traducir los datos

Para trabajar más cómodos, vamos a pasar todo a español.

a) Renombra las columnas: `total_bill` → `cuenta`, `tip` → `propina`, `sex` → `sexo`, `smoker` → `fumador`, `day` → `dia`, `time` → `momento`, `size` → `personas`.

b) Traduce los valores:
- `dia`: `Thur` → `Jueves`, `Fri` → `Viernes`, `Sat` → `Sábado`, `Sun` → `Domingo`
- `momento`: `Lunch` → `Comida`, `Dinner` → `Cena`
- `sexo`: `Female` → `Mujer`, `Male` → `Hombre`
- `fumador`: `Yes` → `sí`, `No` → `no`

Comprueba el resultado con `head()`.

> 💡 **Pista:** `rename(columns={...})` y `.map({...})`.

In [ ]:
propinas = propinas.rename(columns={'total_bill':'cuenta', 'tip':'propina', 'sex':'sexo', 'smoker':'fumador', 'day':'dia', 'time':'momento', 'size':'personas'})
traducciones = {
    'dia': {'Thur':'Jueves', 'Fri':'Viernes', 'Sat':'Sábado', 'Sun':'Domingo'},
    'momento': {'Lunch':'Comida', 'Dinner':'Cena'},
    'sexo': {'Female':'Mujer', 'Male':'Hombre'},
    'fumador': {'Yes':'sí', 'No':'no'}
}
for columna, traduccion in traducciones.items():
    propinas[columna] = propinas[columna].map(traduccion)
print(propinas.head())

### Ejercicio 9 · Nuevas columnas

Una propina de 5 $ no significa lo mismo en una cuenta de 15 $ que en una de 50 $. Por eso vamos a crear variables más útiles:

a) `pct_propina`: la propina como **porcentaje** de la cuenta.

b) `cuenta_por_persona`: lo que pagó cada persona de la mesa.

c) `tamaño_mesa`: agrupa `personas` en `"1-2"`, `"3-4"` y `"5-6"`.

d) `generosa`: `"sí"` si la propina es del 20% o más, `"no"` en caso contrario.

> 💡 **Pistas:** operaciones entre columnas, `pd.cut` (con `bins=[0, 2, 4, 6]`) y `np.where`.

In [ ]:
propinas['pct_propina'] = propinas['propina'] / propinas['cuenta'] * 100
propinas['cuenta_por_persona'] = propinas['cuenta'] / propinas['personas']
propinas['tamaño_mesa'] = pd.cut(propinas['personas'], bins=[0,2,4,6], labels=['1-2','3-4','5-6'])
propinas['generosa'] = np.where(propinas['pct_propina'] >= 20, 'sí', 'no')
print(propinas.head().round(2))

### Ejercicio 10 · Filtrar y ordenar

a) ¿Cuántas mesas dejaron una propina **generosa** (20% o más)? ¿Qué **proporción** del total representan?

b) ¿Cuántas **cenas de fin de semana** (sábado o domingo) fueron de **4 personas o más**? ¿Cuál fue su porcentaje de propina medio?

c) Muestra las **5 mesas** con mayor `pct_propina`, con las columnas `cuenta`, `propina`, `pct_propina`, `personas` y `dia`.

d) Usa `query` para obtener las mesas con **fumadores** y una cuenta **entre 10 y 20 $**. ¿Cuántas son?

> 💡 **Pistas:** filtros con `&`, `isin` y `sort_values`. Dentro de `query` el texto va entre comillas simples: `"fumador == 'sí'"`.

In [ ]:
generosas = propinas['pct_propina'] >= 20
print('a) Generosas:', generosas.sum(), 'Proporción:', generosas.mean(), 'Porcentaje:', round(generosas.mean()*100, 2))
cenas_finde = propinas[(propinas['momento']=='Cena') & propinas['dia'].isin(['Sábado','Domingo']) & (propinas['personas'] >= 4)]
print('b) Mesas:', len(cenas_finde), 'Porcentaje medio:', round(cenas_finde['pct_propina'].mean(), 2))
top5 = propinas.sort_values('pct_propina', ascending=False).head(5)
print('c) Top 5:\n', top5[['cuenta','propina','pct_propina','personas','dia']].round(2))
fumadores_10_20 = propinas.query("fumador == 'sí' and 10 <= cuenta <= 20")
print('d) Mesas:', len(fumadores_10_20))

✍️ **Pregunta:** fíjate en la primera mesa del apartado c). ¿Te parece un caso normal? ¿Cómo crees que afecta a la **media** del porcentaje de propina?

✍️ **Tu respuesta:** 

La primera mesa tiene una cuenta de $7.25 y una propina de $5.15: el 71.03%. Es un valor extremo respecto a las demás mesas, aunque eso no prueba que sea un error. Eleva la media del porcentaje de propina; la mediana suele ser más resistente a valores extremos. En d) se incluyeron ambos límites, $10 y $20.

### Ejercicio 11 · Agrupar y resumir

a) Para cada **día**, calcula el número de mesas, la **media** y la **mediana** de `pct_propina`.

b) Para cada **momento** (comida/cena), crea un resumen con columnas con nombre: `n_mesas`, `cuenta_media`, `propina_media` y `pct_medio`.

c) Para cada número de **personas**, calcula el número de mesas y el `pct_propina` medio.

d) Crea una tabla dinámica con el `pct_propina` medio por **sexo** (filas) y **fumador** (columnas).

e) Quédate solo con las mesas del **domingo** y calcula la media y la mediana de `pct_propina` **con** y **sin** la mesa del 71% que encontraste en el ejercicio 10.

> 💡 **Pistas:** `groupby(...).agg([...])`, `agg(nombre=("columna", "función"))` y `pd.pivot_table`. Redondea con `.round(1)` para que se lea mejor.

In [ ]:
resumen_dia = propinas.groupby('dia')['pct_propina'].agg(['count','mean','median'])
print('a) Por día:\n', resumen_dia.round(1))
resumen_momento = propinas.groupby('momento').agg(n_mesas=('propina','size'), cuenta_media=('cuenta','mean'), propina_media=('propina','mean'), pct_medio=('pct_propina','mean'))
print('b) Por momento:\n', resumen_momento.round(1))
resumen_personas = propinas.groupby('personas').agg(n_mesas=('propina','size'), pct_medio=('pct_propina','mean'))
print('c) Por personas:\n', resumen_personas.round(1))
print('d) Sexo y fumador:\n', propinas.pivot_table(values='pct_propina', index='sexo', columns='fumador', aggfunc='mean').round(1))
domingo = propinas[propinas['dia']=='Domingo']
indice_extremo = top5.index[0]
domingo_sin = domingo.drop(index=indice_extremo)
comparacion = pd.DataFrame({'con_extremo': domingo['pct_propina'].agg(['mean','median']), 'sin_extremo': domingo_sin['pct_propina'].agg(['mean','median'])})
print('e) Domingo:\n', comparacion.round(2))

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Preguntas:**
1. En a), ¿en qué días la media es mayor que la mediana? Según e), ¿cuál de las dos medidas se ve más afectada por una mesa extrema? ¿Cuál usarías para describir la propina "típica"?
2. En c), ¿las mesas más grandes dejan más o menos porcentaje de propina? ¿Te fiarías de los resultados para 1, 5 o 6 personas? ¿Por qué?

✍️ **Tu respuesta:** 

La media supera a la mediana en Domingo, Jueves, Sábado, Viernes. En domingo, al quitar la mesa extrema, la media baja de 16.69% a 15.97%; la mediana cambia de 16.11% a 16.05%. Usaría la mediana para describir la propina típica.
No hay una disminución estricta, pero las mesas de 2 personas promedian 16.57%, frente a 14.59% de las de 4. Hay solo 4 mesas de una persona, 5 mesas de cinco y 4 de seis: sus promedios son poco estables y no bastan para generalizar.

### Ejercicio 12 · Cuidado con las comparaciones

Haz una tabla cruzada que cuente cuántas mesas hay de cada **día** (filas) en cada **momento** (columnas).

> 💡 **Pista:** `pd.crosstab`.

In [ ]:
print(pd.crosstab(propinas['dia'], propinas['momento']))

✍️ **Pregunta:** si comparas las propinas de la comida con las de la cena, ¿estás comparando solo el momento del día, o también otra cosa? ¿Por qué es importante tenerlo en cuenta antes de sacar conclusiones?

✍️ **Tu respuesta:** 

También estamos comparando días distintos. De las 68 comidas, 61 son de jueves y 7 de viernes; las 163 mesas de sábado y domingo son cenas. Momento y día están asociados en esta muestra. No podemos atribuir las diferencias exclusivamente al horario.

### Ejercicio 13 · Combinar tablas

Tenemos otra tabla con información de cada día:
```python
info_dias = pd.DataFrame({
    "dia": ["Jueves", "Viernes", "Sábado", "Domingo"],
    "orden": [4, 5, 6, 7],
    "fin_de_semana": [False, False, True, True],
})
```

a) Añade esta información a `propinas` con `merge`. Comprueba que el número de filas **no cambia**.

b) Compara la **cuenta media** y el **`pct_propina` medio** entre fin de semana y entre semana.

c) Calcula cuánto se facturó en total **cada día**, ordenado cronológicamente (de jueves a domingo).

> 💡 **Pista:** para c), agrupa por `["orden", "dia"]`: `groupby` ordena los grupos por la primera columna.

In [ ]:
info_dias = pd.DataFrame({'dia':['Jueves','Viernes','Sábado','Domingo'], 'orden':[4,5,6,7], 'fin_de_semana':[False,False,True,True]})
n_antes = len(propinas)
propinas = propinas.merge(info_dias, on='dia', how='left', validate='many_to_one')
print('a) Filas antes y después:', n_antes, len(propinas))
assert len(propinas) == n_antes
print('b) Comparación:\n', propinas.groupby('fin_de_semana').agg(cuenta_media=('cuenta','mean'), pct_medio=('pct_propina','mean')).round(2))
print('c) Facturación:\n', propinas.groupby(['orden','dia'])['cuenta'].sum().round(2))

### Ejercicio 14 · De pandas a NumPy

Los modelos de Machine Learning trabajan con arrays de NumPy, así que es habitual pasar de un DataFrame a un array.

a) Crea `X` con las columnas `cuenta`, `propina` y `personas` convertidas a array de NumPy. ¿Qué forma tiene?

b) **Solo con NumPy**, calcula el porcentaje de propina de cada mesa a partir de `X`. Comprueba que su media coincide con la de `propinas["pct_propina"]`.

c) **Solo con NumPy**, ¿cuántas mesas de **2 personas** dejaron **más de 4 $** de propina?

> 💡 **Pistas:** `.to_numpy()`. En `X`, la columna 0 es la cuenta, la 1 la propina y la 2 las personas.

In [ ]:
X = propinas[['cuenta','propina','personas']].to_numpy()
print('a) Forma:', X.shape)
pct_numpy = X[:, 1] / X[:, 0] * 100
print('b) Media NumPy:', pct_numpy.mean(), 'Media pandas:', propinas['pct_propina'].mean())
print('Coinciden:', np.isclose(pct_numpy.mean(), propinas['pct_propina'].mean()))
print('c) Mesas de 2 personas con más de $4:', np.sum((X[:, 2] == 2) & (X[:, 1] > 4)))

### ✍️ Conclusiones de la Parte 2

Imagina que el camarero te pregunta: *"¿qué tengo que hacer para conseguir mejores propinas?"*. Escribe:

1. **Tres hallazgos** sobre las propinas, cada uno apoyado en un número concreto de tus resultados.
2. **Una limitación** de estos datos que haga que debas tener cuidado con tus conclusiones.

✍️ **Tu respuesta:** 

1. 39 de 244 mesas dejan al menos 20% de propina: el 15.98% del total.

2. La propina monetaria media es aproximadamente $3.10 en cena y $2.73 en comida, pero el porcentaje medio es 15.95% y 16.41%, respectivamente. Una propina mayor en dólares no implica un porcentaje mayor.

3. Las mesas de 2 personas dejan un 16.57% medio, frente a 14.59% en las de 4.

Limitación: Son observaciones de un camarero, con tamaños de grupo desiguales y sin variables como calidad del servicio. No permiten afirmar que atender cierto grupo o cambiar de horario cause mejores propinas. Como orientación práctica, conviene distinguir monto y porcentaje y registrar más información antes de tomar decisiones. La media de los porcentajes da el mismo peso a cada mesa; no es lo mismo que dividir la propina total entre la facturación total.

---
## Reto opcional · El Titanic 🚢

En 1912 el Titanic se hundió en su viaje inaugural. El archivo `data/titanic.csv` tiene información de 891 pasajeros. Las columnas que necesitas:

| Columna | Significado |
|---|---|
| `survived` | 1 si sobrevivió, 0 si no |
| `sex` | `male` / `female` |
| `age` | edad (¡algunas faltan!) |
| `fare` | precio del billete |
| `class` | clase del billete: `First`, `Second`, `Third` |

**Pregunta:** ¿se cumplió el famoso *"las mujeres y los niños primero"*? ¿Importaba la clase del billete?

a) Carga los datos y usa `info()` para ver qué columnas tienen valores faltantes.

b) ¿Qué proporción de pasajeros sobrevivió?

c) Calcula la tasa de supervivencia por **sexo**, por **clase** y en una tabla dinámica **sexo × clase**.

d) Agrupa las edades en `"niño"` (0–12), `"adolescente"` (13–18), `"adulto"` (19–60) y `"mayor"` (más de 60), y calcula la tasa de supervivencia y el número de pasajeros de cada grupo.

e) ¿Cuánto costaba de media el billete en cada clase?

> 💡 **Pistas:**
> - La **media** de una columna de 0 y 1 es la **proporción** de unos: `titanic["survived"].mean()` es la tasa de supervivencia.
> - Para d), `pd.cut` con `bins=[0, 12, 18, 60, 100]`. Los pasajeros sin edad (`NaN`) quedan fuera de los grupos automáticamente.

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

In [ ]:
# Tu código aquí

✍️ **Pregunta:** ¿se cumplió *"las mujeres y los niños primero"*? ¿Y la clase, importaba? Responde con números.

✍️ **Tu respuesta:** 